In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, jaccard_score, precision_score, recall_score, f1_score, accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Import utilities from the other files
from data_utils_deep import SegmentationDataset, CLASS_NAMES
from model_utils_deep import get_model, combined_loss, iou_score

# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 10
NUM_EPOCHS = 50
LR = 1e-4
IMAGE_SIZE = 128
NUM_CLASSES = 7

# --- Data Path Placeholders ---
# IMPORTANT: Update this path to your dataset location
BASE_PATH = "../Dataset/Dataset/Prepared_Dataset"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_PATH, "train/masks")
VAL_IMG_DIR = os.path.join(BASE_PATH, "val/images")
VAL_MASK_DIR = os.path.join(BASE_PATH, "val/masks")

# --- Augmentations ---
train_transform = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.RandomRotate90(), A.HorizontalFlip(), A.VerticalFlip(),
    A.Affine(rotate=(-15, 15), scale=(0.9, 1.1), translate_percent=(0.06, 0.06)),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# --- Evaluation Function ---
def validate_epoch(model, val_loader):
    model.eval()
    val_loss_sum, val_miou_sum = 0.0, 0.0
    all_preds, all_masks = [], []
    
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE).squeeze(-1).long()
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = combined_loss(logits, masks)
            
            val_loss_sum += loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1)
            val_miou_sum += iou_score(logits, masks, num_classes=NUM_CLASSES) * imgs.size(0)
            all_preds.append(preds.cpu().numpy())
            all_masks.append(masks.cpu().numpy())

    avg_loss = val_loss_sum / len(val_loader.dataset)
    avg_miou = val_miou_sum / len(val_loader.dataset)
    return avg_loss, avg_miou, np.concatenate(all_preds, axis=0), np.concatenate(all_masks, axis=0)

# --- Plotting Function ---
def plot_results(train_losses, val_losses, val_mious, all_val_preds, all_val_masks, val_dataset, session_title):
    print(f"\n--- Generating Visualizations for: {session_title} ---")
    
    plt.figure(figsize=(14, 6)); plt.suptitle(session_title, fontsize=16)
    plt.subplot(1, 2, 1); plt.plot(train_losses, label='Train Loss'); plt.plot(val_losses, label='Validation Loss'); plt.title('Training & Validation Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2); plt.plot(val_mious, label='Validation mIoU', color='orange'); plt.title('Validation Mean IoU (mIoU)'); plt.xlabel('Epoch'); plt.ylabel('mIoU'); plt.legend(); plt.grid(True)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]); plt.show(block=False)
    
    flat_true, flat_pred = all_val_masks.flatten(), all_val_preds.flatten()
    cm = confusion_matrix(flat_true, flat_pred, labels=np.arange(NUM_CLASSES))
    plt.figure(figsize=(10, 8)); sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES); plt.xlabel('Predicted Label'); plt.ylabel('True Label'); plt.title(f'Confusion Matrix - {session_title}'); plt.show(block=True) # Block here to see this plot before code finishes

# --- Training Session Function ---
def run_training_session(model_name, enhancement_mode, model_save_path):
    print("\n" + "="*60 + f"\nSTARTING TRAINING: Model='{model_name}', Enhancement='{enhancement_mode}'\n" + "="*60 + "\n")

    train_dataset = SegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform, use_enhancement=enhancement_mode)
    val_dataset = SegmentationDataset(VAL_IMG_DIR, VAL_MASK_DIR, transform=val_transform, use_enhancement='none')
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    model = get_model(model_name, DEVICE, classes=NUM_CLASSES)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, mode='max')
    scaler = torch.cuda.amp.GradScaler()

    best_miou = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
    final_val_preds, final_val_masks = None, None

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} (Train)")
        for imgs, masks in progress_bar:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE).squeeze(-1).long()
            optimizer.zero_grad();
            with torch.cuda.amp.autocast(): loss = combined_loss(model(imgs), masks)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            train_loss += loss.item() * imgs.size(0)
            progress_bar.set_postfix(loss=loss.item())

        history['train_loss'].append(train_loss / len(train_loader.dataset))
        val_loss, val_miou, current_preds, current_masks = validate_epoch(model, val_loader)
        history['val_loss'].append(val_loss); history['val_miou'].append(val_miou); scheduler.step(val_miou)
        print(f"Epoch {epoch} finished. Train Loss: {history['train_loss'][-1]:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {val_miou:.4f}")

        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), model_save_path)
            print(f"Model saved to {model_save_path}! (Best mIoU: {best_miou:.4f})")
            final_val_preds, final_val_masks = current_preds, current_masks

    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path)); _, _, final_val_preds, final_val_masks = validate_epoch(model, val_loader)
    plot_results(history['train_loss'], history['val_loss'], history['val_miou'], final_val_preds, final_val_masks, val_dataset, session_title=f"Model: {model_name.upper()} | Enhancement: {enhancement_mode}")

# --- Main Execution ---
def main():
    if not os.path.exists(TRAIN_IMG_DIR) or not os.path.exists(VAL_IMG_DIR):
        print(f"!! ERROR: Data paths incorrect. Expected Prepared_Dataset directory at: {BASE_PATH}")
        return

    # --- DeepLabV3+ Training Sessions ---
    print("--- Starting Training for DeepLabV3+ ---")
    run_training_session('deeplabv3plus', 'none', "deeplab_v3_plus_none_50.pth")
    run_training_session('deeplabv3plus', 'all', "deeplab_v3_plus_preprocess_50.pth")
    run_training_session('deeplabv3plus', 'hybrid', "deeplab_v3_plus_hybrid_50.pth")
    
    print("\nAll requested training sessions completed.")

if __name__ == '__main__':
    try:
        main()
    except Exception as e:
        print(f"\nAn unexpected and fatal error occurred: {e}")

--- Starting Training for DeepLabV3+ ---

STARTING TRAINING: Model='deeplabv3plus', Enhancement='none'

Initializing DeepLabV3+ model with encoder resnet101.


C:\Users\veerk\AppData\Local\Temp\ipykernel_4376\3336034756.py:97: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
c:\Users\veerk\OneDrive\Desktop\DIP Project\dip\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
Epoch 1/50 (Train):   0%|          | 0/713 [00:00<?, ?it/s]c:\Users\veerk\OneDrive\Desktop\DIP Project\dip\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\veerk\AppData\Local\Temp\ipykernel_4376\3336034756.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): loss = combined_loss(model(imgs)

KeyboardInterrupt: 